In [ ]:
import os
import pandas as pd
from pathlib import Path

In [ ]:
# =====================
# CONFIG
# =====================

INPUT_DIR = Path("data/processed/aggregated_by_signal")
OUTPUT_DIR = Path("data/processed/analysis")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =====================
# PHASE DEFINITION (min)
# =====================

def assign_phase(elapsed_min):
    if elapsed_min < 10:
        return "adaptation_seated"
    elif elapsed_min < 20:
        return "standing_pre"
    elif elapsed_min < 30:
        return "low_1"
    elif elapsed_min < 40:
        return "moderate_1"
    elif elapsed_min < 50:
        return "low_2"
    elif elapsed_min < 60:
        return "moderate_2"
    else:
        return "standing_post"

# =====================
# LOAD ALL SIGNAL FILES
# =====================

all_dfs = []

for file in INPUT_DIR.glob("*.csv"):
    df = pd.read_csv(file)

    # Extract signal name from filename
    signal_name = file.stem.lower()  # es: t_abdomen

    df["signal"] = signal_name

    all_dfs.append(df)

# Merge all
df_long = pd.concat(all_dfs, ignore_index=True)

# =====================
# CLEAN + FEATURES
# =====================

# Ensure correct types
df_long["elapsed_minutes"] = df_long["elapsed_minutes"].astype(float)

# Assign phase
df_long["phase"] = df_long["elapsed_minutes"].apply(assign_phase)

# 5-min bins
df_long["time_bin_5min"] = (df_long["elapsed_minutes"] // 5) * 5

# =====================
# SAVE LONG FORMAT
# =====================

df_long.to_csv(OUTPUT_DIR / "all_signals_long.csv", index=False)

# =====================
# CREATE WIDE DATASET
# =====================

df_wide = df_long.pivot_table(
    index=["subject_id", "condition", "timestamp", "elapsed_seconds", "elapsed_minutes", "phase", "time_bin_5min"],
    columns="signal",
    values="value"
).reset_index()

# =====================
# ADD DERIVED VARIABLES
# =====================

# Microclimate gradients
if "t_microclimateback" in df_wide.columns and "t_microclimatefront" in df_wide.columns:
    df_wide["t_microclimate_gradient"] = (
        df_wide["t_microclimateback"] - df_wide["t_microclimatefront"]
    )

if "rh_microclimateback" in df_wide.columns and "rh_microclimatefront" in df_wide.columns:
    df_wide["rh_microclimate_gradient"] = (
        df_wide["rh_microclimateback"] - df_wide["rh_microclimatefront"]
    )

# =====================
# SAVE WIDE DATASET
# =====================

df_wide.to_csv(OUTPUT_DIR / "analysis_dataset_wide.csv", index=False)

print("Analysis dataset created successfully")